In [1]:

import sys
import os
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
from PIL import Image
import base64
import io
import json
from datetime import datetime
import uuid

import time
import re

# Add backend directory to Python path
backend_path = Path(".").resolve().parent / "backend"
sys.path.insert(0, str(backend_path))
sys.path.insert(0, str(Path(".").resolve().parent))

# Load environment variables
load_dotenv()

# Initialize Gemini client
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_IMAGE_API_KEY = os.getenv("GEMINI_IMAGE_API_KEY")

if not GEMINI_API_KEY:
    print("❌ GEMINI_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요.")
    sys.exit()
else:
    print("✅ Gemini API 키 로드 완료")

if not GEMINI_IMAGE_API_KEY:
    print("❌ GEMINI_IMAGE_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요.")
    sys.exit()
else:
    print("✅ Gemini Image API 키 로드 완료")

✅ Gemini API 키 로드 완료
✅ Gemini Image API 키 로드 완료


In [2]:
print("\n" + "=" * 100)
print("🎯 NameTag 브랜드 가이드 생성")
print("=" * 100)

# 사용자 입력 받기
print("\n📝 브랜드 정보를 입력해주세요:\n")

business_type = "20대를 위한 감성 소품 온라인 셀렉샵"  # Simulated input
# business_type = input("🏢 업종/서비스를 입력하세요 (예: 온라인 친환경 쇼핑몰): ").strip()
while len(business_type) < 3:
    business_type = input("❌ 최소 3글자 이상 입력해주세요: ").strip()

vibes_input = "따뜻한, 감성적인, 자연친화적, 레트로한"  # Simulated input
# vibes_input = input("✨ 브랜드 감성을 선택하세요 (쉼표로 구분, 예: 모던, 신뢰): ").strip()
vibes = [v.strip() for v in vibes_input.split(",") if v.strip()]
while len(vibes) == 0 or len(vibes) > 4:
    vibes_input = input("❌ 1~4개의 감성을 선택해주세요: ").strip()
    vibes = [v.strip() for v in vibes_input.split(",") if v.strip()]

target = "30대 직장 여성, 소소한 취미생활을 즐기고 자신만의 공간을 꾸미는 것에 관심 있는 분들"  # Simulated input
# target = input("👥 타겟 고객을 입력하세요 (예: 20-40대 환경 의식 있는 소비자): ").strip()
while len(target) < 3:
    target = input("❌ 최소 3글자 이상 입력해주세요: ").strip()

keywords = "따뜻함, 일상, 발견"  # Simulated input
# keywords = input("🔑 추가 키워드 (선택사항): ").strip()

print("\n" + "=" * 100)
print("✅ 입력된 브랜드 정보:")
print(f"업종/서비스: {business_type}")
print(f"브랜드 감성: {', '.join(vibes)}")
print(f"타겟 고객: {target}")
print(f"추가 키워드: {keywords}")


🎯 NameTag 브랜드 가이드 생성

📝 브랜드 정보를 입력해주세요:


✅ 입력된 브랜드 정보:
업종/서비스: 20대를 위한 감성 소품 온라인 셀렉샵
브랜드 감성: 따뜻한, 감성적인, 자연친화적, 레트로한
타겟 고객: 30대 직장 여성, 소소한 취미생활을 즐기고 자신만의 공간을 꾸미는 것에 관심 있는 분들
추가 키워드: 따뜻함, 일상, 발견


In [3]:
# Gemini API에 보낼 프롬프트 생성 함수
def get_prompt(prompt, business_type, vibes, target, selected_brand_name="브랜드명 미정"):
    # 2️⃣ Gemini API에 보낼 프롬프트 생성
    print("\n📨 Gemini API에 요청할 프롬프트:")
    prompt = f"""
[기본 입력 정보]
- 업종/서비스: {business_type}
- 브랜드 감성: {', '.join(vibes)}
- 타겟 고객: {target}
- 확정된 브랜드명: {selected_brand_name}
""" + prompt
    print(prompt)
    print("\n" + "-" * 100)

    return prompt

# Gemini API에 요청하는 함수
def request_gemini_api(prompt, SYSTEM_PROMPT, max_retries=3):
    print("\n" + "-" * 100)
    print("💬 AI에 요청 중입니다... 잠시만 기다려주세요.")
    print("-" * 100)
    
    client = genai.Client(api_key=GEMINI_API_KEY)
    
    retries = 0
    while retries <= max_retries:
        try:
            response = client.models.generate_content(
                model="gemini-3-flash-preview",
                contents=prompt,
                config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT),
            )
            raw_text = response.text or ""
            print("\n✅ AI 응답 수신 완료!")
            print(f"응답 텍스트: {raw_text[:100]}...") # 너무 길 수 있어 100자만 자름
            return raw_text
        
        except Exception as e:
            error_str = str(e)
            # 발생한 에러 메시지 안에 429, 500, 503이 포함되어 있는지 확인
            if any(code in error_str for code in ["429", "500", "503"]):
                retries += 1
                print(f"\n⚠️ 서버 과부하 또는 할당량 초과 에러 발생: {error_str.split('.')[0]}")
                
                if retries <= max_retries:
                    print(f"⏳ 1분 30초 대기 후 재요청합니다... (재시도 횟수: {retries}/{max_retries})")
                    time.sleep(90) # 90초(1분 30초) 대기
                    continue # while 루프의 처음으로 돌아가서 다시 try 시도
                else:
                    print("\n❌ 최대 재시도 횟수를 초과했습니다. 잠시 후 다시 프로그램을 실행해주세요.")
                    sys.exit()
            else:
                # 429, 500, 503 외의 알 수 없는 다른 에러인 경우 즉시 종료
                print(f"\n❌ 처리할 수 없는 AI 응답 오류: {e}")
                sys.exit()

# 4AI 응답 파싱
def parse_ai_response(raw_text):
    try:
        # JSON 추출
        import re
        json_match = re.search(r'\{[\s\S]*\}', raw_text)
        if json_match:
            ai_response = json.loads(json_match.group())
        else:
            ai_response = json.loads(raw_text)
        
        print("✅ JSON 파싱 완료!")
        print(f"파싱된 JSON: {json.dumps(ai_response)}")
        return ai_response
        
    except Exception as e:
        print(f"❌ JSON 파싱 오류: {e}")
        print(f"원본 응답: {raw_text}")
        ai_response = {}
        sys.exit()

# 사용자 입력 처리용 헬퍼 함수
def get_user_choice(prompt_text, max_num):
    choice = input(prompt_text).strip()
    try:
        index = int(choice) - 1
        if 0 <= index < max_num:
            return index
        else:
            print(f"⚠️ 1~{max_num} 사이의 숫자를 입력해주세요. 기본값(1)을 선택합니다.")
            return 0
    except ValueError:
        print("⚠️ 잘못된 입력입니다. 기본값(1)을 선택합니다.")
        return 0

# 아이템 선택용 헬퍼 함수
def select_item(step_title, items, format_func, prompt_text):
    print(f"\n[{step_title}]")
    for i, item in enumerate(items):
        # format_func(lambda)를 통해 원하는 출력 형태로 포매팅
        print(f"  {i+1}. {format_func(item)}")
    return get_user_choice(prompt_text, len(items))

In [5]:
print("Part 1 실행")

SYSTEM_PROMPT = """당신은 10년 경력의 브랜드 디렉터입니다.
초기 창업자에게 최소 브랜드(MVB: Minimum Viable Brand)를 제공하는 것이 목표입니다.
반드시 JSON만 출력하고 마크다운 코드블록이나 설명 텍스트를 절대 포함하지 마세요.
각 후보는 서로 완전히 다른 방향성과 감성을 가져야 합니다."""

prompt_0 = f"""
아래 JSON 형식으로만 초안 후보 4개를 응답하세요:

{{
  "candidates": [
    {{
      "id": 1,
      "brand_name": "브랜드명 (국문, 2~20글자)",
      "brand_name_en": "영문명",
      "name_meaning": "이름의 의미와 어감 설명 (50자 이내)",
      "slogan": "핵심 슬로건 (국문, 15자 이내)",
      "story_summary": "브랜드 탄생 서사 요약 (100자 이내)",
      "seed_color": "#XXXXXX",
      "seed_color_reason": "이 컬러를 선택한 이유 (30자 이내)"
    }},
    {{
      "id": 2,
      "brand_name": "...",
      "brand_name_en": "...",
      "name_meaning": "...",
      "slogan": "...",
      "story_summary": "...",
      "seed_color": "...",
      "seed_color_reason": "..."
    }},
    {{
      "id": 3,
      "brand_name": "...",
      "brand_name_en": "...",
      "name_meaning": "...",
      "slogan": "...",
      "story_summary": "...",
      "seed_color": "...",
      "seed_color_reason": "..."
    }},
    {{
      "id": 4,
      "brand_name": "...",
      "brand_name_en": "...",
      "name_meaning": "...",
      "slogan": "...",
      "story_summary": "...",
      "seed_color": "...",
      "seed_color_reason": "..."
    }}
  ]
}}"""

prompt_0 = get_prompt(prompt_0, business_type, vibes, target)

Part 1 실행

📨 Gemini API에 요청할 프롬프트:

[기본 입력 정보]
- 업종/서비스: 20대를 위한 감성 소품 온라인 셀렉샵
- 브랜드 감성: 따뜻한, 감성적인, 자연친화적, 레트로한
- 타겟 고객: 30대 직장 여성, 소소한 취미생활을 즐기고 자신만의 공간을 꾸미는 것에 관심 있는 분들
- 확정된 브랜드명: 브랜드명 미정

아래 JSON 형식으로만 초안 후보 4개를 응답하세요:

{
  "candidates": [
    {
      "id": 1,
      "brand_name": "브랜드명 (국문, 2~20글자)",
      "brand_name_en": "영문명",
      "name_meaning": "이름의 의미와 어감 설명 (50자 이내)",
      "slogan": "핵심 슬로건 (국문, 15자 이내)",
      "story_summary": "브랜드 탄생 서사 요약 (100자 이내)",
      "seed_color": "#XXXXXX",
      "seed_color_reason": "이 컬러를 선택한 이유 (30자 이내)"
    },
    {
      "id": 2,
      "brand_name": "...",
      "brand_name_en": "...",
      "name_meaning": "...",
      "slogan": "...",
      "story_summary": "...",
      "seed_color": "...",
      "seed_color_reason": "..."
    },
    {
      "id": 3,
      "brand_name": "...",
      "brand_name_en": "...",
      "name_meaning": "...",
      "slogan": "...",
      "story_summary": "...",
      "seed_color": "...",
      "seed_color_reason"

In [6]:
raw_text_0 = request_gemini_api(prompt_0, SYSTEM_PROMPT)

parsed_response_0 = parse_ai_response(raw_text_0)


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

✅ AI 응답 수신 완료!
응답 텍스트: {
  "candidates": [
    {
      "id": 1,
      "brand_name": "온온",
      "brand_name_en": "ONON",
  ...
✅ JSON 파싱 완료!
파싱된 JSON: {"candidates": [{"id": 1, "brand_name": "\uc628\uc628", "brand_name_en": "ONON", "name_meaning": "\ub530\ub73b\ud568(\u6eab)\uc774 \uacb9\uce5c\ub2e4\ub294 \ub73b\uc73c\ub85c \uacf5\uac04\uc758 \uc628\uae30\ub97c \uc0c1\uc9d5", "slogan": "\ub2f9\uc2e0\uc758 \uacf5\uac04\uc5d0 \uba38\ubb34\ub294 \ub2e4\uc815\ud55c \uc628\uae30", "story_summary": "\uc9c0\uce5c \ud558\ub8e8\uc758 \ub05d, \uc9d1\uc73c\ub85c \ub3cc\uc544\uc628 \ub2f9\uc2e0\uc744 \uc548\uc544\uc8fc\ub294 \ub530\uc2a4\ud558\uace0 \ubd80\ub4dc\ub7ec\uc6b4 \uc18c\ud488\ub4e4\uc744 \ud050\ub808\uc774\uc158\ud569\ub2c8\ub2e4.", "seed_color": "#EADBC8", 

In [7]:
# 3️⃣ 결과 표시
print("\n" + "=" * 100)
print("🎨 AI가 제안한 4가지 브랜드 옵션")
print("=" * 100)

for candidate in parsed_response_0.get("candidates", []):
    print(f"\n옵션 {candidate['id']}:")
    print(f"브랜드명: {candidate['brand_name']} ({candidate['brand_name_en']})")
    print(f"이름의 의미: {candidate['name_meaning']}")
    print(f"슬로건: {candidate['slogan']}")
    print(f"브랜드 스토리 요약: {candidate['story_summary']}")
    print(f"시드 컬러: {candidate['seed_color']} (이유: {candidate['seed_color_reason']})")
    print(f"시드 컬러 이유 : {candidate['seed_color_reason']}")


🎨 AI가 제안한 4가지 브랜드 옵션

옵션 1:
브랜드명: 온온 (ONON)
이름의 의미: 따뜻함(溫)이 겹친다는 뜻으로 공간의 온기를 상징
슬로건: 당신의 공간에 머무는 다정한 온기
브랜드 스토리 요약: 지친 하루의 끝, 집으로 돌아온 당신을 안아주는 따스하고 부드러운 소품들을 큐레이션합니다.
시드 컬러: #EADBC8 (이유: 포근함과 안락함을 주는 베이지)
시드 컬러 이유 : 포근함과 안락함을 주는 베이지

옵션 2:
브랜드명: 숲결 (SUPGYEOL)
이름의 의미: 숲의 숨결과 질감을 일상으로 가져온다는 의미
슬로건: 내 방에서 만나는 작은 숲
브랜드 스토리 요약: 인위적이지 않은 자연의 결을 담은 소품으로 도심 속 나만의 고요한 휴식처를 완성합니다.
시드 컬러: #829460 (이유: 자연의 평온함을 담은 올리브 그린)
시드 컬러 이유 : 자연의 평온함을 담은 올리브 그린

옵션 3:
브랜드명: 오후의 갈피 (AFTERNOON PAUSE)
이름의 의미: 바쁜 일상 중 책갈피처럼 꽂아두는 쉼표 같은 시간
슬로건: 멈춰버린 시간, 나만의 아날로그
브랜드 스토리 요약: 레트로한 감성의 소품들을 통해 잊고 있던 순수한 기억과 정적인 여유를 선물합니다.
시드 컬러: #B05B3B (이유: 시간의 흔적이 묻은 테라코타)
시드 컬러 이유 : 시간의 흔적이 묻은 테라코타

옵션 4:
브랜드명: 소소조각 (SOSO PIECE)
이름의 의미: 작고 소중한 취향의 조각들이 모여 완성되는 일상
슬로건: 소소한 취미가 나의 일상이 될 때
브랜드 스토리 요약: 나만의 공간을 채우는 작은 오브제들로 직장 여성의 생산적이고 아름다운 취미 생활을 응원합니다.
시드 컬러: #A7BBC7 (이유: 차분하고 지적인 느낌의 뮤트 블루)
시드 컬러 이유 : 차분하고 지적인 느낌의 뮤트 블루


In [8]:
candidates = parsed_response_0.get("candidates", [])
num_candidates = len(candidates)

print("\n" + "=" * 60)
print("🎯 [토스(Toss) 방식 브랜딩] 각 항목별로 마음에 드는 조합을 만들어보세요!")
print("=" * 60)

# 3. 통합 함수(select_item)를 활용하여 선택 진행
name_idx = select_item(
    "1. 브랜드 이름 후보", 
    candidates, 
    lambda c: f"{c['brand_name']} ({c['brand_name_en']})", 
    "▶ 마음에 드는 브랜드 이름의 번호를 선택하세요: "
)

meaning_idx = select_item(
    "2. 네이밍 의미 후보", 
    candidates, 
    lambda c: c['name_meaning'], 
    "▶ 마음에 드는 의미의 번호를 선택하세요: "
)

slogan_idx = select_item(
    "3. 핵심 슬로건 후보", 
    candidates, 
    lambda c: c['slogan'], 
    "▶ 마음에 드는 슬로건의 번호를 선택하세요: "
)

story_idx = select_item(
    "4. 스토리 요약 후보", 
    candidates, 
    lambda c: c['story_summary'], 
    "▶ 마음에 드는 스토리의 번호를 선택하세요: "
)

color_idx = select_item(
    "5. 브랜드 컬러 후보", 
    candidates, 
    lambda c: f"{c['seed_color']} ({c['seed_color_reason']})", 
    "▶ 마음에 드는 컬러의 번호를 선택하세요: "
)

# 최종 선택 결과 조합 딕셔너리 생성
create_brand = {
    "brand_name": candidates[name_idx]["brand_name"],
    "brand_name_en": candidates[name_idx]["brand_name_en"],
    "name_meaning": candidates[meaning_idx]["name_meaning"],
    "slogan": candidates[slogan_idx]["slogan"],
    "story_summary": candidates[story_idx]["story_summary"],
    "seed_color": candidates[color_idx]["seed_color"],
    "seed_color_reason": candidates[color_idx]["seed_color_reason"]
}

# 결과 출력
print("\n" + "=" * 60)
print("🎉 [최종 브랜드 조합이 완성되었습니다!]")
print("-" * 60)
print(f"🔹 이름   : {create_brand['brand_name']} ({create_brand['brand_name_en']})")
print(f"🔹 의미   : {create_brand['name_meaning']}")
print(f"🔹 슬로건 : {create_brand['slogan']}")
print(f"🔹 스토리 : {create_brand['story_summary']}")
print(f"🔹 컬러   : {create_brand['seed_color']} ({create_brand['seed_color_reason']})")
print("=" * 60)


🎯 [토스(Toss) 방식 브랜딩] 각 항목별로 마음에 드는 조합을 만들어보세요!

[1. 브랜드 이름 후보]
  1. 온온 (ONON)
  2. 숲결 (SUPGYEOL)
  3. 오후의 갈피 (AFTERNOON PAUSE)
  4. 소소조각 (SOSO PIECE)

[2. 네이밍 의미 후보]
  1. 따뜻함(溫)이 겹친다는 뜻으로 공간의 온기를 상징
  2. 숲의 숨결과 질감을 일상으로 가져온다는 의미
  3. 바쁜 일상 중 책갈피처럼 꽂아두는 쉼표 같은 시간
  4. 작고 소중한 취향의 조각들이 모여 완성되는 일상

[3. 핵심 슬로건 후보]
  1. 당신의 공간에 머무는 다정한 온기
  2. 내 방에서 만나는 작은 숲
  3. 멈춰버린 시간, 나만의 아날로그
  4. 소소한 취미가 나의 일상이 될 때

[4. 스토리 요약 후보]
  1. 지친 하루의 끝, 집으로 돌아온 당신을 안아주는 따스하고 부드러운 소품들을 큐레이션합니다.
  2. 인위적이지 않은 자연의 결을 담은 소품으로 도심 속 나만의 고요한 휴식처를 완성합니다.
  3. 레트로한 감성의 소품들을 통해 잊고 있던 순수한 기억과 정적인 여유를 선물합니다.
  4. 나만의 공간을 채우는 작은 오브제들로 직장 여성의 생산적이고 아름다운 취미 생활을 응원합니다.

[5. 브랜드 컬러 후보]
  1. #EADBC8 (포근함과 안락함을 주는 베이지)
  2. #829460 (자연의 평온함을 담은 올리브 그린)
  3. #B05B3B (시간의 흔적이 묻은 테라코타)
  4. #A7BBC7 (차분하고 지적인 느낌의 뮤트 블루)

🎉 [최종 브랜드 조합이 완성되었습니다!]
------------------------------------------------------------
🔹 이름   : 소소조각 (SOSO PIECE)
🔹 의미   : 바쁜 일상 중 책갈피처럼 꽂아두는 쉼표 같은 시간
🔹 슬로건 : 멈춰버린 시간, 나만의 아날로그
🔹 스토리 : 지친 하루의 끝, 집으로 돌아온 당신을 안아주는 따스하고 부

In [9]:
create_brand

{'brand_name': '소소조각',
 'brand_name_en': 'SOSO PIECE',
 'name_meaning': '바쁜 일상 중 책갈피처럼 꽂아두는 쉼표 같은 시간',
 'slogan': '멈춰버린 시간, 나만의 아날로그',
 'story_summary': '지친 하루의 끝, 집으로 돌아온 당신을 안아주는 따스하고 부드러운 소품들을 큐레이션합니다.',
 'seed_color': '#A7BBC7',
 'seed_color_reason': '차분하고 지적인 느낌의 뮤트 블루'}

In [11]:
prompt_A_1 = f"""사용자가 이전 단계에서 선택한 핵심 브랜드 정체성을 바탕으로 브랜드 가이드의 '철학과 가치' 영역을 생성하세요.

[사용자 최종 선택값 - 이 값들은 절대 변경하지 말고 그대로 발전시켜야 합니다]
- 이름 선택: {create_brand['brand_name']} ({create_brand['brand_name_en']})
- 의미 선택: {create_brand['name_meaning']}
- 슬로건 선택: {create_brand['slogan']}
- 스토리 선택: {create_brand['story_summary']}
- 컬러 선택: {create_brand['seed_color']} ({create_brand['seed_color_reason']})

[작성 가이드라인]
1. 철학 선언문(manifesto): 단순 소개가 아닌, 이 브랜드가 세상이 어떻게 되어야 한다고 믿는지 세계관을 감성적 언어로 1~2문단 작성하세요. (창업자의 신념, 고객 삶에 대한 통찰 포함)
2. 존재 이유(raison_detre): 이 브랜드가 세상에 없었다면 무엇이 부족했을지 1문장으로 정의하세요.
3. 브랜드 에센스: 1~3단어의 키워드로 압축하고, 이것이 실제 제품/콘텐츠/응대에서 어떻게 구현되는지 3~4줄로 설명하세요.
4. 한 줄 정의(brand_definition): 30자 이내로, 이름과 함께 읽힐 때 즉시 이해되도록 작성하세요. (예: 퇴근 후 나를 위한 감성 소품 큐레이션)
5. 핵심 가치: 3~5개를 제시하고, 각 가치가 실제로 어떻게 구현되는지 1줄 예시를 반드시 포함하세요.

아래 JSON 형식에 맞추어 응답을 생성하세요:

{{
  "brand_philosophy": {{
    "manifesto": "철학 선언문 (1~2문단)",
    "raison_detre": "브랜드 존재 이유 (1문장)",
    "stand_against": ["반대하는 것 1", "반대하는 것 2", "반대하는 것 3"],
    "stand_for": ["지지하고 실천하는 가치 1", "지지하고 실천하는 가치 2", "지지하고 실천하는 가치 3"]
  }},
  "brand_essence": {{
    "keyword": "에센스 키워드 (1~3단어)",
    "keyword_explanation": "해당 키워드의 실제 구현 설명 (3~4줄)",
    "brand_definition": "브랜드 한 줄 정의 (30자 이내)"
  }},
  "core_identity": {{
    "mission": "브랜드 미션 (동사형, '우리는 누구에게 무엇을 어떻게 제공한다' 형태)",
    "vision": "브랜드 비전 (명사형, 5~10년 후의 모습)",
    "core_values": [
      {{ 
        "value": "핵심 가치 1 (키워드)", 
        "description": "20자 이내 설명", 
        "example": "실제 구현 예시 1줄" 
      }},
      {{ 
        "value": "핵심 가치 2 (키워드)", 
        "description": "20자 이내 설명", 
        "example": "실제 구현 예시 1줄" 
      }},
      {{ 
        "value": "핵심 가치 3 (키워드)", 
        "description": "20자 이내 설명", 
        "example": "실제 구현 예시 1줄" 
      }}
    ]
  }}
}}"""

prompt_A_2 = f"""사용자가 이전 단계에서 선택한 핵심 데이터를 바탕으로 브랜드 가이드의 '네이밍, 슬로건, 브랜드 스토리' 영역을 확장 생성하세요.

[사용자 최종 선택값 - 이 값들은 절대 변경하지 말고 그대로 발전시켜야 합니다]
- 이름 선택: {create_brand['brand_name']} ({create_brand['brand_name_en']})
- 의미 선택: {create_brand['name_meaning']}
- 슬로건 선택: {create_brand['slogan']}
- 스토리 선택: {create_brand['story_summary']}
- 컬러 선택: {create_brand['seed_color']} ({create_brand['seed_color_reason']})

[작성 가이드라인]
1. 네이밍: 확정된 브랜드명의 영문명과 함께, 실제 사용 가능성이 높은 도메인(예: todaysondo.com) 및 SNS 핸들(예: @todays_ondo)을 추천하세요. 대안 네이밍 2개를 추가로 제안하세요.
2. 이름 탄생 스토리: 브랜드명이 만들어진 배경을 100자 이상의 서사형으로 작성하세요.
3. 슬로건: 확정된 국문 슬로건을 유지하되 영문 슬로건을 추가하고, 상황별(예: 제품 소개, 포장/배송 등)로 사용할 서브 슬로건 2~3개를 작성하세요.
4. 브랜드 스토리(매우 중요): 확정된 스토리 요약을 바탕으로 '탄생 배경 → 고객의 현실 → 브랜드의 약속 → 함께하는 미래'의 서사 구조를 갖추어 작성하세요. 일상의 구체적인 장면(예: 매일 반복되는 출퇴근, 숨 가쁘게 돌아가는 업무 시간...)에서 시작하는 서사로, 반드시 3~5문단 분량, 총 300자 이상으로 작성해야 합니다.

아래 JSON 형식에 맞추어 응답을 생성하세요:

{{
  "naming": {{
    "brand_name": "{create_brand['brand_name']}",
    "brand_name_en": "영문명 (도메인/SNS 사용 가능 형태)",
    "naming_rationale": "왜 이 이름인지 선택 근거 (3~5줄)",
    "name_story": "이름 탄생 스토리 (반드시 100자 이상)",
    "domain_suggestions": ["추천도메인1.com", "추천도메인2.co.kr", "추천도메인3.shop"],
    "sns_handles": ["@추천핸들1", "@추천핸들2"],
    "alternative_names": [
      {{ "name": "대안명1", "name_en": "Alt1", "rationale": "선택 근거 (3~5줄)" }},
      {{ "name": "대안명2", "name_en": "Alt2", "rationale": "선택 근거 (3~5줄)" }}
    ]
  }},
  "slogans": {{
    "main_slogan": "{create_brand['slogan']}",
    "main_slogan_en": "영문 메인 슬로건",
    "slogan_rationale": "메인 슬로건 의도 설명 (2~3줄)",
    "sub_slogans": [
      {{ "slogan": "서브 슬로건 1", "emphasis": "사용 상황 및 강조점 (예: 상세페이지 도입부)" }},
      {{ "slogan": "서브 슬로건 2", "emphasis": "사용 상황 및 강조점 (예: 패키징 동봉 엽서)" }},
      {{ "slogan": "서브 슬로건 3", "emphasis": "사용 상황 및 강조점" }}
    ]
  }},
  "brand_story": "브랜드 스토리 전문 (탄생 배경, 고객 현실, 약속, 미래 구조. 반드시 300자 이상, 3~5문단)"
}}"""

prompt_A_3 = f"""사용자가 이전 단계에서 선택한 핵심 데이터를 바탕으로 브랜드 가이드의 '포지셔닝 및 브랜드 약속' 영역을 생성하세요.

[사용자 최종 선택값 - 이 값들은 절대 변경하지 말고 그대로 발전시켜야 합니다]
- 이름 선택: {create_brand['brand_name']} ({create_brand['brand_name_en']})
- 의미 선택: {create_brand['name_meaning']}
- 슬로건 선택: {create_brand['slogan']}
- 스토리 선택: {create_brand['story_summary']}
- 컬러 선택: {create_brand['seed_color']} ({create_brand['seed_color_reason']})

[작성 가이드라인]
1. 포지셔닝 선언문: "For [타겟], [브랜드명]은 [경쟁군]와 달리 [차별화 가치]를 제공합니다." 형태를 정확히 지켜 1문장으로 작성하세요.
2. 차별화 포인트: 경쟁사 대비 우리 브랜드가 다른 점을 구체적인 항목 3~5가지로 제시하세요.
3. 포지셔닝 맵: 브랜드를 시장에 위치시킬 2개의 축(예: 고감성↔저감성, 저가↔프리미엄 등 업종에 맞는 축)을 설정하고 우리 브랜드의 위치를 설명하세요.
4. 브랜드 약속 선언: "우리는 당신에게 항상 [____]을 드립니다." 형태를 지켜 1문장으로 작성하세요. 이 약속이 채널별(마케팅, 패키징, CS 등)로 어떻게 실제로 구현되는지 예시 3가지를 함께 제시하세요.

아래 JSON 형식에 맞추어 응답을 생성하세요:

{{
  "positioning": {{
    "statement": "For [타겟], [{create_brand['brand_name']}]은(는) [경쟁군]와(과) 달리 [차별화 가치]를 제공합니다.",
    "differentiators": [
      {{ "point": "차별화 포인트 1 제목", "description": "구체적인 차별점 설명" }},
      {{ "point": "차별화 포인트 2 제목", "description": "구체적인 차별점 설명" }},
      {{ "point": "차별화 포인트 3 제목", "description": "구체적인 차별점 설명" }}
    ],
    "positioning_map": {{
      "axis_x": "X축 설명 (예: 접근성/가격 등)",
      "axis_y": "Y축 설명 (예: 감성/전문성 등)",
      "our_position": "해당 축을 기준으로 한 우리 브랜드의 위치 설명"
    }}
  }},
  "brand_promise": {{
    "declaration": "우리는 당신에게 항상 [빈칸 채움]을(를) 드립니다.",
    "implementations": [
      {{ "channel": "마케팅 메시지", "example": "적용 예시 1줄" }},
      {{ "channel": "패키징/언박싱", "example": "적용 예시 1줄" }},
      {{ "channel": "고객 응대(CS)", "example": "적용 예시 1줄" }}
    ]
  }}
}}"""

prompt_A_1 = get_prompt(prompt_A_1, business_type, vibes, target, create_brand['brand_name'])
prompt_A_2 = get_prompt(prompt_A_2, business_type, vibes, target, create_brand['brand_name'])
prompt_A_3 = get_prompt(prompt_A_3, business_type, vibes, target, create_brand['brand_name'])


📨 Gemini API에 요청할 프롬프트:

[기본 입력 정보]
- 업종/서비스: 20대를 위한 감성 소품 온라인 셀렉샵
- 브랜드 감성: 따뜻한, 감성적인, 자연친화적, 레트로한
- 타겟 고객: 30대 직장 여성, 소소한 취미생활을 즐기고 자신만의 공간을 꾸미는 것에 관심 있는 분들
- 확정된 브랜드명: 소소조각
사용자가 이전 단계에서 선택한 핵심 브랜드 정체성을 바탕으로 브랜드 가이드의 '철학과 가치' 영역을 생성하세요.

[사용자 최종 선택값 - 이 값들은 절대 변경하지 말고 그대로 발전시켜야 합니다]
- 이름 선택: 소소조각 (SOSO PIECE)
- 의미 선택: 바쁜 일상 중 책갈피처럼 꽂아두는 쉼표 같은 시간
- 슬로건 선택: 멈춰버린 시간, 나만의 아날로그
- 스토리 선택: 지친 하루의 끝, 집으로 돌아온 당신을 안아주는 따스하고 부드러운 소품들을 큐레이션합니다.
- 컬러 선택: #A7BBC7 (차분하고 지적인 느낌의 뮤트 블루)

[작성 가이드라인]
1. 철학 선언문(manifesto): 단순 소개가 아닌, 이 브랜드가 세상이 어떻게 되어야 한다고 믿는지 세계관을 감성적 언어로 1~2문단 작성하세요. (창업자의 신념, 고객 삶에 대한 통찰 포함)
2. 존재 이유(raison_detre): 이 브랜드가 세상에 없었다면 무엇이 부족했을지 1문장으로 정의하세요.
3. 브랜드 에센스: 1~3단어의 키워드로 압축하고, 이것이 실제 제품/콘텐츠/응대에서 어떻게 구현되는지 3~4줄로 설명하세요.
4. 한 줄 정의(brand_definition): 30자 이내로, 이름과 함께 읽힐 때 즉시 이해되도록 작성하세요. (예: 퇴근 후 나를 위한 감성 소품 큐레이션)
5. 핵심 가치: 3~5개를 제시하고, 각 가치가 실제로 어떻게 구현되는지 1줄 예시를 반드시 포함하세요.

아래 JSON 형식에 맞추어 응답을 생성하세요:

{
  "brand_philosophy": {
    "manifesto": "철학 선언문 (1~2문단)",
    "raison_detre":

In [12]:
raw_text_A_1 = request_gemini_api(prompt_A_1, SYSTEM_PROMPT)
raw_text_A_2 = request_gemini_api(prompt_A_2, SYSTEM_PROMPT)
raw_text_A_3 = request_gemini_api(prompt_A_3, SYSTEM_PROMPT)

parsed_response_A_1 = parse_ai_response(raw_text_A_1)
parsed_response_A_2 = parse_ai_response(raw_text_A_2)
parsed_response_A_3 = parse_ai_response(raw_text_A_3)


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

✅ AI 응답 수신 완료!
응답 텍스트: {
  "brand_philosophy": {
    "manifesto": "우리는 속도가 미덕인 세상에서 잠시 멈춰 서는 일의 위대함을 믿습니다. 쉼표가 없는 문장은 끝내 숨이...

----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

✅ AI 응답 수신 완료!
응답 텍스트: {
  "naming": {
    "brand_name": "소소조각",
    "brand_name_en": "sosopiece (sosopiece.com)",
    "nam...

----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

✅ AI 응답 수신 완료!
응답 텍스트: {
  "positioning": {
   

In [13]:
# 결과 출력
print("\n" + "=" * 60)
print("브랜드 철학")
print(f"브랜드 철학 선언문 : {parsed_response_A_1['brand_philosophy']['manifesto']}")
print(f"브랜드 존재 이유 : {parsed_response_A_1['brand_philosophy']['raison_detre']}")
print(f"브랜드가 반대하는 것 : {', '.join(parsed_response_A_1['brand_philosophy']['stand_against'])}")
print(f"브랜드가 지지하는 것 : {', '.join(parsed_response_A_1['brand_philosophy']['stand_for'])}")

print("\n" + "=" * 60)
print("브랜드 에센스")
print(f"브랜드 에센스 키워드 : {parsed_response_A_1['brand_essence']['keyword']}")
print(f"브랜드 에센스 설명 : {parsed_response_A_1['brand_essence']['keyword_explanation']}")
print(f"브랜드 한 줄 정의 : {parsed_response_A_1['brand_essence']['brand_definition']}")

print("\n" + "=" * 60)
print("브랜드 핵심 정체성")
print(f"브랜드 미션 : {parsed_response_A_1['core_identity']['mission']}")
print(f"브랜드 비전 : {parsed_response_A_1['core_identity']['vision']}")
print("브랜드 핵심 가치:")
for value in parsed_response_A_1['core_identity']['core_values']:
    print(f"  - 가치: {value['value']}, 설명: {value['description']}, 예시: {value['example']}")

print("\n" + "=" * 60)
print("네이밍 및 슬로건")
print(f"확정된 브랜드명: {parsed_response_A_2['naming']['brand_name']} ({parsed_response_A_2['naming']['brand_name_en']})")
print(f"네이밍 선택 근거: {parsed_response_A_2['naming']['naming_rationale']}")
print(f"이름 탄생 스토리: {parsed_response_A_2['naming']['name_story']}")
print(f"추천 도메인: {', '.join(parsed_response_A_2['naming']['domain_suggestions'])}")
print(f"추천 SNS 핸들: {', '.join(parsed_response_A_2['naming']['sns_handles'])}")
print("대안 네이밍:")
for alt in parsed_response_A_2['naming']['alternative_names']:
    print(f"  - 이름: {alt['name']} ({alt['name_en']}), 선택 근거: {alt['rationale']}")

print(f"확정된 슬로건: {parsed_response_A_2['slogans']['main_slogan']} ({parsed_response_A_2['slogans']['main_slogan_en']})")
print(f"슬로건 의도 설명: {parsed_response_A_2['slogans']['slogan_rationale']}")
print("서브 슬로건:") 
for sub in parsed_response_A_2['slogans']['sub_slogans']:
    print(f"  - 슬로건: {sub['slogan']}, 사용 상황: {sub['emphasis']}")
print(f"브랜드 스토리: {parsed_response_A_2['brand_story']}")


print("\n" + "=" * 60)
print("포지셔닝 및 브랜드 약속")
print(f"포지셔닝 선언문: {parsed_response_A_3['positioning']['statement']}")
print("차별화 포인트:")
for point in parsed_response_A_3['positioning']['differentiators']:
    print(f"  - {point['point']}: {point['description']}")
print(f"포지셔닝 맵: X축 - {parsed_response_A_3['positioning']['positioning_map']['axis_x']}, Y축 - {parsed_response_A_3['positioning']['positioning_map']['axis_y']}")
print(f"우리 브랜드의 위치 설명: {parsed_response_A_3['positioning']['positioning_map']['our_position']}")
print(f"브랜드 약속 선언: {parsed_response_A_3['brand_promise']['declaration']}")
print("브랜드 약속 구현 예시:")
for impl in parsed_response_A_3['brand_promise']['implementations']:
    print(f"  - 채널: {impl['channel']}, 예시: {impl['example']}")

create_brand["philosophy"]    = parsed_response_A_1["brand_philosophy"]
create_brand["essence"]       = parsed_response_A_1["brand_essence"]
create_brand["core_identity"] = parsed_response_A_1["core_identity"]
create_brand["naming"]        = parsed_response_A_2["naming"]
create_brand["slogans"]       = parsed_response_A_2["slogans"]
create_brand["brand_story"]   = parsed_response_A_2["brand_story"]
create_brand["positioning"]   = parsed_response_A_3["positioning"]
create_brand["brand_promise"] = parsed_response_A_3["brand_promise"]


브랜드 철학
브랜드 철학 선언문 : 우리는 속도가 미덕인 세상에서 잠시 멈춰 서는 일의 위대함을 믿습니다. 쉼표가 없는 문장은 끝내 숨이 차듯, 우리 삶에도 일상의 여백을 표시할 '책갈피'가 필요합니다. 소소조각은 단순히 물건을 파는 것이 아니라, 당신의 공간에 아주 작은 아날로그의 리듬을 심어 하루의 끝이 온전한 위로가 되길 바랍니다.

지친 하루 끝에 마주하는 부드러운 질감과 차분한 색조는 무너졌던 마음의 균형을 되찾아주는 조용한 힘이 됩니다. 우리는 모두가 자신만의 속도로 머무를 수 있는 권리가 있다고 믿으며, 그 머무름의 시간을 가장 따뜻한 조각들로 채워나가고자 합니다.
브랜드 존재 이유 : 쉼표 없이 흘러가는 삭막한 도시 생활 속에서 나를 되찾아주는 유일한 아날로그적 도피처가 되기 위해 존재합니다.
브랜드가 반대하는 것 : 공장식 대량 생산의 차가움, 효율성만을 강조하는 디지털 라이프, 유행에 따라 금방 버려지는 소비 문화
브랜드가 지지하는 것 : 손길이 닿은 듯한 따스한 질감, 세월의 흔적이 느껴지는 레트로한 감성, 환경을 생각하는 지속 가능한 소재

브랜드 에센스
브랜드 에센스 키워드 : 아날로그적 쉼표
브랜드 에센스 설명 : 뮤트 블루 컬러의 차분함과 자연 친화적 소재를 결합하여 시각적, 촉각적 편안함을 제공합니다. 제품 배치부터 패키징까지 모든 과정에서 사용자가 물리적으로 '시간이 멈춘 듯한' 평온함을 느끼도록 설계하며, 이를 통해 일상 속에 명확한 휴식의 지점을 만들어냅니다.
브랜드 한 줄 정의 : 지친 일상을 위로하는 아날로그적 쉼표, 소소조각

브랜드 핵심 정체성
브랜드 미션 : 우리는 일상의 여유가 필요한 30대 여성에게 따스한 온기를 담은 소품을 제공하여 그들만의 아날로그적 휴식 시간을 완성한다.
브랜드 비전 : 모든 이의 공간에 마음을 토닥이는 따뜻한 조각 하나가 깃드는 라이프스타일 브랜드.
브랜드 핵심 가치:
  - 가치: 정서적 온기, 설명: 마음을 안아주는 부드러움, 예시: 제품 검수 시 시각적인 아름다움뿐만 아니라 손 끝에 

In [ ]:
create_brand

{'brand_name': '소소조각',
 'brand_name_en': 'SOSO PIECE',
 'name_meaning': '바쁜 일상 중 책갈피처럼 꽂아두는 쉼표 같은 시간',
 'slogan': '멈춰버린 시간, 나만의 아날로그',
 'story_summary': '지친 하루의 끝, 집으로 돌아온 당신을 안아주는 따스하고 부드러운 소품들을 큐레이션합니다.',
 'seed_color': '#A7BBC7',
 'seed_color_reason': '차분하고 지적인 느낌의 뮤트 블루',
 'philosophy': {'manifesto': "우리는 속도가 미덕인 세상에서 잠시 멈춰 서는 일의 위대함을 믿습니다. 쉼표가 없는 문장은 끝내 숨이 차듯, 우리 삶에도 일상의 여백을 표시할 '책갈피'가 필요합니다. 소소조각은 단순히 물건을 파는 것이 아니라, 당신의 공간에 아주 작은 아날로그의 리듬을 심어 하루의 끝이 온전한 위로가 되길 바랍니다.\n\n지친 하루 끝에 마주하는 부드러운 질감과 차분한 색조는 무너졌던 마음의 균형을 되찾아주는 조용한 힘이 됩니다. 우리는 모두가 자신만의 속도로 머무를 수 있는 권리가 있다고 믿으며, 그 머무름의 시간을 가장 따뜻한 조각들로 채워나가고자 합니다.",
  'raison_detre': '쉼표 없이 흘러가는 삭막한 도시 생활 속에서 나를 되찾아주는 유일한 아날로그적 도피처가 되기 위해 존재합니다.',
  'stand_against': ['공장식 대량 생산의 차가움',
   '효율성만을 강조하는 디지털 라이프',
   '유행에 따라 금방 버려지는 소비 문화'],
  'stand_for': ['손길이 닿은 듯한 따스한 질감',
   '세월의 흔적이 느껴지는 레트로한 감성',
   '환경을 생각하는 지속 가능한 소재']},
 'essence': {'keyword': '아날로그적 쉼표',
  'keyword_explanation': "뮤트 블루 컬러의 차분함과 자연 친화적 소재를 결합하여 시각적, 촉각적 편안함을 제공합니다. 제품 배

: 